# Laboratorio 7: ejecuciones en Colab

Función: preparar GPU, ejecutar los apartados 2–4 y generar el Markdown del apartado 5.

Selecciona una GPU en la configuración del entorno de ejecución. El código comprobará CUDA. Las cinco semillas son cinco ejecuciones completas por variante, cada una de 300.000 pasos. El estudio completo suma 21 ejecuciones; puede durar varias sesiones.

Este cuaderno es nuevo; `RunDDPG_Colab.ipynb` permanece intacto.

In [ ]:
# TODO (preparación): trabajar únicamente en la carpeta Lab7 de esta sesión.
#______________________________
from pathlib import Path
import os
from google.colab import files

carpeta = Path('/content/Lab7')
carpeta.mkdir(exist_ok=True)
os.chdir(carpeta)
# Sube los archivos .py de Lab7; puedes añadir LEEME.md y este cuaderno.
subidos = files.upload()
print('Directorio de trabajo:', Path.cwd())
#______________________________


In [ ]:
# TODO (dependencias): instalar Box2D según el cuaderno original.
#______________________________
%pip install -q swig
%pip install -q "gymnasium[box2d]" matplotlib numpy

import torch
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
#______________________________


In [ ]:
# TODO (validación): comprobar fórmulas y reanudación antes de entrenar.
#______________________________
import subprocess
import sys
subprocess.run([sys.executable, 'Verificar.py'], check=True)
#______________________________


## Elegir el trabajo

`all` ejecuta DDPG/semilla 0 y las cuatro variantes TD3 con cinco semillas. Para repartir el trabajo entre recursos independientes usa `ddpg`, `full`, `no_minimum`, `no_smoothing` o `no_delay`, con carpetas diferentes. También puedes repartir `SEMILLAS`. Mantén T=300000 y la misma configuración. No se garantiza que varios procesos en una misma GPU sean más rápidos.

In [ ]:
# TODO (apartados 2 y 4): declarar el presupuesto antes de medir.
#______________________________
VARIANTE = 'all'
SEMILLAS = [0, 1, 2, 3, 4]
T = 300000
SALIDA = 'resultados'
comando = [sys.executable, 'Experimentos.py', '--variant', VARIANTE,
           '--seeds', *map(str, SEMILLAS), '--steps', str(T), '--output', SALIDA]
subprocess.run(comando, check=True)
#______________________________


## Conservar resultados de cada recurso

Descarga el archivo antes de terminar la sesión. Incluye checkpoints para poder conservar el agente completo. Para reunir recursos distintos, descomprime sus carpetas dentro de tu `Lab7` local conservando nombres de salida diferentes. Los checkpoints no necesitan incluirse en la entrega final.

In [ ]:
# TODO (persistencia): descargar los datos y checkpoints de este trabajo.
#______________________________
import zipfile
salida = (carpeta / SALIDA).resolve()
assert salida.is_relative_to(carpeta)
archivo = carpeta / f'{salida.name}_datos.zip'
with zipfile.ZipFile(archivo, 'w', compression=zipfile.ZIP_DEFLATED) as paquete:
    for ruta in salida.rglob('*'):
        if ruta.is_file():
            paquete.write(ruta, ruta.relative_to(carpeta))
files.download(str(archivo))
#______________________________


## Generar informe y entrega

Ejecuta estas celdas cuando estén reunidas las 21 ejecuciones. Cambia `ENTRADAS` si repartiste el trabajo. El generador rechaza datos ausentes o incompletos y genera únicamente Markdown, PNG y CSV. Comprueba las cuatro páginas al convertir el Markdown a PDF.

In [ ]:
# TODO (apartado 5): generar figuras, tabla e informe con mediciones reales.
#______________________________
ENTRADAS = ['resultados']
subprocess.run([sys.executable, 'Diagnosticos.py', '--input', *ENTRADAS], check=True)
#______________________________


In [ ]:
# TODO (entrega): empaquetar código, Markdown, figuras y mediciones.
#______________________________
assert (carpeta / 'resumen_mediciones.csv').is_file(), 'Genera primero el informe final.'
rutas = set(carpeta.glob('*.py')) | set(carpeta.glob('*.md')) | set(carpeta.glob('*.ipynb'))
rutas.add(carpeta / 'resumen_mediciones.csv')
rutas.update((carpeta / 'graficas').glob('*.png'))
for entrada in ENTRADAS:
    base = (carpeta / entrada).resolve()
    assert base.is_relative_to(carpeta)
    rutas.update(base.glob('*/*.csv'))
    rutas.update(base.glob('*/configuracion.json'))
entrega = carpeta / 'Entrega_Lab7.zip'
with zipfile.ZipFile(entrega, 'w', compression=zipfile.ZIP_DEFLATED) as paquete:
    for ruta in sorted(rutas):
        paquete.write(ruta, ruta.relative_to(carpeta))
files.download(str(entrega))
#______________________________
